# Lending Club Credit Risk Analysis
## Notebook 1 — Data Preparation

**Author:** Jeffrey Symons  
**Date:** July 2026  
**Purpose:** Load, filter, clean, optimize, and save the Lending Club dataset for EDA and modeling.

### What this notebook does:
1. Inspects the raw 1.1GB CSV using DuckDB without loading into memory
2. Filters to resolved individual loans and selects bureau attributes
3. Investigates vintage coverage issues and removes affected columns/rows
4. Optimizes dtypes for memory efficiency (936MB → 259MB)
5. Treats outliers with data-driven cutoffs
6. Saves clean dataset to Parquet for fast repeated access

**Key decision:** Joint applications (1.8% of resolved loans) excluded — without joint
income and DTI fields, including them adds noise without predictive value.

## 1. Imports

In [1]:
import duckdb
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_rows', 100)

print('Libraries loaded successfully')

Libraries loaded successfully


## 2. Inspect Raw File with DuckDB

DuckDB queries the CSV directly without loading it into memory.
The raw file is ~1.1GB / 2.6M rows — too large for pandas directly.

In [2]:
# Row count
duckdb.query("SELECT COUNT(*) as total_rows FROM 'lending_club.csv'").df()

,total_rows
0,2260668


In [3]:
# Loan status distribution
duckdb.query("""
    SELECT loan_status, COUNT(*) as cnt
    FROM 'lending_club.csv'
    GROUP BY loan_status
    ORDER BY cnt DESC
""").df()

,loan_status,cnt
0,Fully Paid,1041952
1,Current,919695
2,Charged Off,261655
3,Late (31-120 days),21897
4,In Grace Period,8952
5,Late (16-30 days),3737
6,Does not meet the credit policy. Status:Fully ...,1988
7,Does not meet the credit policy. Status:Charge...,761
8,Default,31


In [4]:
# Application type distribution
duckdb.query("""
    SELECT application_type, COUNT(*) as cnt
    FROM 'lending_club.csv'
    WHERE loan_status IN ('Charged Off', 'Fully Paid')
    GROUP BY application_type
    ORDER BY cnt DESC
""").df()

# Joint applications = 1.8% of resolved loans — excluded

,application_type,cnt
0,Individual,1280370
1,Joint App,23237


## 3. Load Data with DuckDB

Selected 75 columns based on data dictionary review:
- Bureau attributes known at origination
- Excluded post-origination variables (data leakage)
- Excluded Lending Club derived labels (grade, int_rate — circular)
- Excluded FICO (not present in this dataset version)

In [5]:
df = duckdb.query('''
SELECT  acc_open_past_24mths,
        addr_state,
        all_util,
        annual_inc,
        avg_cur_bal,
        bc_open_to_buy,
        bc_util,
        collections_12_mths_ex_med,
        delinq_2yrs,
        dti,
        earliest_cr_line,
        emp_length,
        funded_amnt,
        grade,
        home_ownership,
        il_util,
        inq_fi,
        inq_last_12m,
        inq_last_6mths,
        int_rate,
        issue_d,
        loan_amnt,
        loan_status,
        max_bal_bc,
        mo_sin_old_il_acct,
        mo_sin_old_rev_tl_op,
        mo_sin_rcnt_rev_tl_op,
        mo_sin_rcnt_tl,
        mort_acc,
        mths_since_last_delinq,
        mths_since_last_major_derog,
        mths_since_last_record,
        mths_since_rcnt_il,
        mths_since_recent_bc,
        mths_since_recent_bc_dlq,
        mths_since_recent_inq,
        mths_since_recent_revol_delinq,
        num_accts_ever_120_pd,
        num_actv_bc_tl,
        num_actv_rev_tl,
        num_bc_sats,
        num_bc_tl,
        num_il_tl,
        num_op_rev_tl,
        num_rev_accts,
        num_rev_tl_bal_gt_0,
        num_sats,
        num_tl_90g_dpd_24m,
        num_tl_op_past_12m,
        open_acc,
        open_acc_6m,
        open_il_12m,
        open_il_24m,
        open_rv_12m,
        open_rv_24m,
        pct_tl_nvr_dlq,
        percent_bc_gt_75,
        pub_rec,
        pub_rec_bankruptcies,
        purpose,
        revol_bal,
        revol_util,
        sub_grade,
        tax_liens,
        term,
        tot_coll_amt,
        tot_cur_bal,
        tot_hi_cred_lim,
        total_acc,
        total_bal_ex_mort,
        total_bal_il,
        total_bc_limit,
        total_il_high_credit_limit,
        total_rev_hi_lim,
        verification_status
FROM 'lending_club.csv'
WHERE loan_status IN ('Charged Off', 'Fully Paid')
AND application_type = 'Individual'
''').df()

print('Shape after load:', df.shape)
print('Memory:', df.memory_usage(deep=True).sum() / 1e6, 'MB')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape after load: (1280370, 75)
Memory: 936.389536 MB


## 4. Vintage Coverage Analysis

High-null columns investigated for vintage patterns.
Two groups identified with data collection cutoffs.

In [6]:
# Check null percentages
null_pct = df.isnull().sum() / len(df) * 100
high_null = null_pct[null_pct > 3].sort_values(ascending=False)
print(high_null)

mths_since_last_record           83.01
mths_since_recent_bc_dlq         76.25
mths_since_last_major_derog      73.70
il_util                          67.71
mths_since_recent_revol_delinq   66.54
mths_since_rcnt_il               63.74
all_util                         62.78
open_acc_6m                      62.78
inq_last_12m                     62.78
open_rv_24m                      62.78
open_il_12m                      62.78
open_il_24m                      62.78
open_rv_12m                      62.78
max_bal_bc                       62.78
inq_fi                           62.78
total_bal_il                     62.78
mths_since_last_delinq           50.44
mths_since_recent_inq            13.07
mo_sin_old_il_acct                8.09
pct_tl_nvr_dlq                    5.29
avg_cur_bal                       5.28
mo_sin_old_rev_tl_op              5.27
mo_sin_rcnt_rev_tl_op             5.27
num_rev_accts                     5.27
mo_sin_rcnt_tl                    5.27
total_rev_hi_lim         

In [7]:
# Check vintage cutoff for high-null columns
high_null_cols = [
    'all_util', 'inq_fi', 'inq_last_12m', 'max_bal_bc', 'open_acc_6m',
    'open_il_12m', 'open_il_24m', 'open_rv_12m', 'open_rv_24m',
    'total_bal_il', 'il_util', 'mths_since_rcnt_il'
]

results = []
for col in high_null_cols:
    vintage_nulls = df.groupby('issue_d')[col].apply(
        lambda x: x.isnull().sum() / len(x) * 100
    ).reset_index()
    vintage_nulls.columns = ['issue_d', 'null_pct']
    vintage_nulls['issue_d'] = pd.to_datetime(
        vintage_nulls['issue_d'].astype(str), format='mixed'
    )
    vintage_nulls = vintage_nulls.sort_values('issue_d')
    first_available = vintage_nulls[
        vintage_nulls['null_pct'] < 100]['issue_d'].min()
    null_pct_overall = df[col].isnull().sum() / len(df) * 100
    results.append({
        'column': col,
        'null_pct': round(null_pct_overall, 2),
        'first_available': first_available
    })

pd.DataFrame(results).sort_values('null_pct', ascending=False)

,column,null_pct,first_available
10,il_util,67.71,2015-12-01
11,mths_since_rcnt_il,63.74,2015-12-01
0,all_util,62.78,2015-12-01
1,inq_fi,62.78,2015-12-01
3,max_bal_bc,62.78,2015-12-01
2,inq_last_12m,62.78,2015-12-01
4,open_acc_6m,62.78,2015-12-01
5,open_il_12m,62.78,2015-12-01
7,open_rv_12m,62.78,2015-12-01
6,open_il_24m,62.78,2015-12-01


In [8]:
# Drop Dec 2015 vintage coverage columns
# These were not collected by Lending Club until Dec 2015
# Including them introduces vintage bias into the model
vintage_2015_cols = [
    'all_util', 'inq_fi', 'inq_last_12m', 'max_bal_bc', 'open_acc_6m',
    'open_il_12m', 'open_il_24m', 'open_rv_12m', 'open_rv_24m',
    'total_bal_il', 'il_util', 'mths_since_rcnt_il'
]
df = df.drop(columns=vintage_2015_cols)
print('Shape after vintage column drops:', df.shape)

Shape after vintage column drops: (1280370, 63)


In [9]:
# Drop pre-Aug 2012 rows
# 35 bureau attributes not collected until Aug 2012
# Trade: lose 4.92% of rows, gain nearly double the feature set
df['issue_d_dt'] = pd.to_datetime(df['issue_d'].astype(str), format='mixed')

pre_2012 = df[df['issue_d_dt'] < '2012-08-01']
print(f'Rows before Aug 2012: {len(pre_2012):,} ({len(pre_2012)/len(df)*100:.2f}%)')

df = df[df['issue_d_dt'] >= '2012-08-01']
df = df.drop(columns=['issue_d_dt'])
print('Shape after row filter:', df.shape)

Rows before Aug 2012: 62,936 (4.92%)


Shape after row filter: (1217434, 63)


## 5. Column Cleaning

In [10]:
# Clean term column — remove 'months' text, convert to Int8
df['term'] = df['term'].str.replace('months', '').str.strip().astype('Int8')
print(df['term'].value_counts())

term
36    926395
60    291039
Name: count, dtype: Int64


In [11]:
# Convert date columns
df['issue_d'] = pd.to_datetime(df['issue_d'].astype(str), format='mixed')
df['earliest_cr_line'] = pd.to_datetime(
    df['earliest_cr_line'].astype(str), format='mixed'
)

# Engineer credit age — months of credit history at origination
# Capped at 480 months (40 years) to remove reversal effect in older borrowers
df['credit_age_months'] = (
    (df['issue_d'] - df['earliest_cr_line']).dt.days / 30
).round().astype(int)
df['credit_age_months'] = df['credit_age_months'].clip(upper=480)

print(df['credit_age_months'].describe())

count   1217434.00
mean        198.72
std          89.13
min          37.00
25%         138.00
50%         181.00
75%         246.00
max         480.00
Name: credit_age_months, dtype: float64


## 6. Dtype Optimization

All dtypes assigned based on confirmed min/max values.
Nullable types (Int8, Int16, Float32) used to preserve meaningful nulls.

In [12]:
# Review min/max before assigning dtypes
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'min_values': df.min(numeric_only=True),
    'max_values': df.max(numeric_only=True)
})
print(summary)

                                         dtype  null_pct  min_values  \
acc_open_past_24mths                     Int64      0.00        0.00   
addr_state                                 str      0.00        <NA>   
annual_inc                             float64      0.00     2400.00   
avg_cur_bal                              Int64      0.38        0.00   
bc_open_to_buy                           Int64      1.04        0.00   
bc_util                                float64      1.10        0.00   
collections_12_mths_ex_med               Int64      0.00        0.00   
credit_age_months                        int64      0.00       37.00   
delinq_2yrs                              int64      0.00        0.00   
dti                                    float64      0.00       -1.00   
earliest_cr_line                datetime64[us]      0.00        <NA>   
emp_length                                 str      0.00        <NA>   
funded_amnt                              int64      0.00     100

In [13]:
# Int8 — confirmed max values all under 127
int8_cols = [
    'acc_open_past_24mths',       # max 64
    'collections_12_mths_ex_med', # max 20
    'delinq_2yrs',                # max 39
    'inq_last_6mths',             # max 8
    'mort_acc',                   # max 51
    'mths_since_recent_inq',      # max 25
    'num_accts_ever_120_pd',      # max 51
    'num_actv_bc_tl',             # max 35
    'num_actv_rev_tl',            # max 63
    'num_bc_sats',                # max 63
    'num_bc_tl',                  # max 70
    'num_op_rev_tl',              # max 83
    'num_rev_tl_bal_gt_0',        # max 45
    'num_sats',                   # max 90
    'num_tl_90g_dpd_24m',         # max 39
    'num_tl_op_past_12m',         # max 32
    'open_acc',                   # max 90
    'pub_rec',                    # max 86
    'pub_rec_bankruptcies',       # max 12
    'tax_liens'                   # max 85
]

# Int16 — exceeds Int8 range
int16_cols = [
    'credit_age_months',              # max 480
    'mo_sin_old_il_acct',            # max 999
    'mo_sin_old_rev_tl_op',          # max 852
    'mo_sin_rcnt_rev_tl_op',         # max 438
    'mo_sin_rcnt_tl',                # max 314
    'mths_since_last_delinq',        # max 226
    'mths_since_last_major_derog',   # max 226
    'mths_since_last_record',        # max 123
    'mths_since_recent_bc',          # max 639
    'mths_since_recent_bc_dlq',      # max 202
    'mths_since_recent_revol_delinq', # max 202
    'num_il_tl',                     # max 159
    'num_rev_accts',                 # max 128
    'total_acc'                      # max 176
]

# Int32 — large dollar amounts and counts
int32_cols = [
    'avg_cur_bal',                # max 958,084
    'bc_open_to_buy',             # max 559,912
    'funded_amnt',                # max 40,000
    'loan_amnt',                  # max 40,000
    'revol_bal',                  # max 2,904,836
    'total_bal_ex_mort',          # max 3,408,095
    'total_bc_limit',             # max 1,105,500
    'total_il_high_credit_limit', # max 2,101,913
    'tot_coll_amt',               # max 9,152,545
    'tot_cur_bal',                # max 8,000,078
    'tot_hi_cred_lim',            # max 9,999,999
    'total_rev_hi_lim'            # max 9,999,999
]

for col in int8_cols:
    df[col] = df[col].astype('Int8')
for col in int16_cols:
    df[col] = df[col].astype('Int16')
for col in int32_cols:
    df[col] = df[col].astype('Int32')

print('Memory after int optimization:', 
      df.memory_usage(deep=True).sum() / 1e6, 'MB')

Memory after int optimization: 419.318558 MB


In [14]:
# Float32 — sufficient precision for all float columns
float32_cols = [
    'annual_inc', 'bc_util', 'dti', 'int_rate',
    'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'revol_util'
]
for col in float32_cols:
    df[col] = df[col].astype('Float32')

print('Memory after float optimization:', 
      df.memory_usage(deep=True).sum() / 1e6, 'MB')

Memory after float optimization: 393.752444 MB


In [15]:
# Category — low cardinality string columns
cat_cols = [
    'addr_state', 'emp_length', 'grade', 'home_ownership',
    'loan_status', 'purpose', 'sub_grade', 'verification_status'
]
for col in cat_cols:
    df[col] = df[col].astype('category')

print('Memory after category optimization:', 
      df.memory_usage(deep=True).sum() / 1e6, 'MB')
print('Final shape:', df.shape)

Memory after category optimization: 256.880119 MB
Final shape: (1217434, 64)


## 7. Outlier Treatment

Outliers identified via min/max inspection and bucketing analysis.
bc_util and revol_util capped at 200% — values 100-200% are legitimate
(bureau credit limit reporting gaps, over-limit spending by card issuers).
Only values above 200% treated as data errors.

In [16]:
print('Rows before outlier treatment:', len(df))

# Preserve nulls explicitly — pandas comparison operators drop NaN rows
df = df[(df['dti'] >= 0) | (df['dti'].isnull())]
df = df[(df['revol_util'] <= 200) | (df['revol_util'].isnull())]
df = df[(df['bc_util'] <= 200) | (df['bc_util'].isnull())]

print('Rows after outlier treatment:', len(df))
print('Rows removed:', 1217434 - len(df))

Rows before outlier treatment: 1217434


Rows after outlier treatment: 1217422
Rows removed: 12


## 8. Target Variable and Save

In [17]:
# Create binary target variable
df['default'] = (df['loan_status'] == 'Charged Off').astype('Int8')

print('Default rate:', df['default'].mean().round(4))
print(df['default'].value_counts())

Default rate: 0.2024
default
0    971074
1    246348
Name: count, dtype: Int64


In [18]:
# Save to parquet — loads in seconds on future sessions
df.to_parquet('lending_club_v2.parquet', index=False, engine='fastparquet')

print('Saved successfully')
print('Final shape:', df.shape)
print('Final memory:', df.memory_usage(deep=True).sum() / 1e6, 'MB')

Saved successfully
Final shape: (1217422, 65)
Final memory: 259.312539 MB


## Summary

| Stage | Memory |
|---|---|
| Raw CSV | ~1,100 MB |
| After DuckDB filter + column selection | 936 MB |
| After vintage column drops + row filter | 770 MB |
| After integer optimization | 419 MB |
| After float optimization | 394 MB |
| After category optimization | **259 MB** |

**Total reduction: 76% from post-load size**

Load for future sessions:
```python
df = pd.read_parquet('lending_club_v2.parquet', engine='fastparquet')
```